# Curvature Comparative

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path


def plot_nemo_curvatures_final(root_dir, save_path=None):
    # Professional publication settings
    plt.rcdefaults()
    sns.set_theme(style="white")
    plt.rcParams.update({
        "font.family": "Arial",
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "pdf.fonttype": 42,
        "axes.linewidth": 0.8,
        "grid.color": "#f0f0f0"
    })

    data_list = []
    root = Path(root_dir)
    series_order = ["shells_shifted", "cylinder_diagonal", "dumbbell_bridge"]

    for folder in root.iterdir():
        if not folder.is_dir() or folder.name.startswith('.'): continue
        for csv_file in folder.glob("*.csv"):
            if "curv_r" not in csv_file.name: continue

            # Use raw strings for labels to avoid regex issues later
            c_type = "Gaussian ($K$)" if "gauss" in csv_file.name else "Mean ($H$)"
            layer = "Inner" if "inner" in csv_file.name else "Outer"

            df_temp = pd.read_csv(csv_file)
            data_list.append(pd.DataFrame({
                "Geometry": folder.name.replace('_', '\n').title(),
                "Layer": layer,
                "Type": c_type,
                "Curvature": df_temp.iloc[:, 0].values
            }))

    df = pd.concat(data_list, ignore_index=True)
    order_mapped = [s.replace('_', '\n').title() for s in series_order]
    palette = {"Outer": "#2c7bb6", "Inner": "#d7191c"}

    fig, axes = plt.subplots(1, 2, figsize=(7, 4))

    # Configuration for panels
    panel_info = [
        {"type_key": "Mean ($H$)", "title": "Mean Curvature ($H$)", "unit": r"($\mu m^{-1}$)"},
        {"type_key": "Gaussian ($K$)", "title": "Gaussian Curvature ($K$)", "unit": r"($\mu m^{-2}$)"}
    ]

    for i, info in enumerate(panel_info):
        ax = axes[i]
        subset = df[df["Type"] == info["type_key"]]

        sns.boxplot(
            data=subset, x="Geometry", y="Curvature", hue="Layer",
            hue_order=["Inner", "Outer"], order=order_mapped,
            palette=palette, width=0.6,
            linewidth=0.8,
            showfliers=True,
            fliersize=1.5,
            flierprops={
                "marker": "o",
                "markerfacecolor": "none",
                "markeredgecolor": "black",
                "alpha": 0.3,
                "markeredgewidth": 0.4
            },
            ax=ax
        )

        ax.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.4, zorder=0)
        ax.set_ylabel(f"{info['title']}\n{info['unit']}")
        ax.set_xlabel("")

        sns.despine(ax=ax, offset=5)
        ax.grid(axis='y', linestyle=':', color='#dddddd', alpha=0.8, zorder=-1)

        if ax.get_legend():
            ax.get_legend().remove()

    # Place legend above both plots
    handles, labels_lg = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels_lg, title="Layer", loc='upper center',
               bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=True)

    plt.tight_layout()

    if save_path:
        out_path = Path(save_path) / "nemo_morphology_final.pdf"
        plt.savefig(out_path, bbox_inches='tight', dpi=600)

    plt.show()


# Execution
root_path = "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_CURVATURES"
plot_nemo_curvatures_final(root_path, save_path=root_path)

# Thickness + Field

In [ ]:
from module_scripts.analysis import spherical_project, spherical_project_vectors
from module_scripts.visuals import plot_spherical_projection, color_scalar
from module_scripts.datahandler import load_mesh, load_array
import os

In [ ]:
heatmappath = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted'
layer_mesh = load_mesh(
    filepath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/inner_mesh_smooth_subset_proj_0_to_9_um_mean/layer_mesh.ply')
directors_2dcurved_avg = load_array(name="directors-avg_2dcurved_r-20.0um",
                                    folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/inner_mesh_smooth_subset_proj_0_to_9_um_mean/')

idxs_sel = load_array("calcindeces",
                      folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/inner_mesh_smooth_subset_proj_0_to_9_um_mean/').astype(
    int)
thickness_vals = load_array("inner_mesh_smooth_subset_VS_outer_mesh_smooth_subset_thickness",
                            folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/')
s_2dcurv = load_array("S-order_2dcurved_r-20.0um",
                      folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/inner_mesh_smooth_subset_proj_0_to_9_um_mean/')

pol_vecfield = load_array(name=f"def-pol_2dcurved",
                          folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/inner_mesh_smooth_subset_proj_0_to_9_um_mean/')
charge_pol_linked_idxs = load_array(name=f"def-pol_2dcurved_idxs",
                                    folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/inner_mesh_smooth_subset_proj_0_to_9_um_mean/').astype(
    int)
m_charge = load_array(name=f"top-charge_2dcurved",
                      folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/inner_mesh_smooth_subset_proj_0_to_9_um_mean/')
defect_idxs_calc = load_array(name=f"top-charge_2dcurved_idxs",
                              folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/inner_mesh_smooth_subset_proj_0_to_9_um_mean/').astype(
    int)

sph_proj_phi, sph_proj_theta = spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                       directors_2dcurved_avg[:, 3:])
pol_dir_phi, pol_dir_theta = spherical_project_vectors(pol_vecfield[:, :3], pol_vecfield[:, 3:])

from scipy.spatial import KDTree

tree = KDTree(layer_mesh.vertices[idxs_sel])
distances, pol_idxs = tree.query(pol_vecfield[:, :3])

plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=thickness_vals, cmap="RdPu",
                          vec_pos_phi=sph_proj_phi[idxs_sel], vec_pos_theta=sph_proj_theta[idxs_sel],
                          vec_dir_phi=vec_dir_phi, vec_dir_theta=vec_dir_theta, veccolor="k", alpha=1.0,
                          scale_factor=5, arrow_alpha=0.8, vec_manual_vminmax=[0, 1], vec_width=0.002,
                          cmap_label=r"Thickness $d$ ($\mu$m)", interp_grid_n=1000,
                          savefig=os.path.join(heatmappath, f"spherical_projection_field_avg-nematic.pdf"),
                          figsize=(16, 6), marker_idxs=idxs_sel[defect_idxs_calc],
                          marker_color=color_scalar(m_charge, manual_vminmax=[-1, 1], cmap="rainbow"),
                          marker_vec=(pol_dir_phi, pol_dir_theta, idxs_sel[pol_idxs]),
                          marker_vec_color=color_scalar(m_charge[charge_pol_linked_idxs], manual_vminmax=[-1, 1],
                                                        cmap="rainbow"), marker_alpha=0.95)

# RENAME previous convention NEMO layer folders

In [ ]:
import os

root_dir = '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/'

print(">> STARTING FOLDER RENAMING...")

for root, dirs, files in os.walk(root_dir):
    for dir_name in dirs:
        # Identify target folders
        if dir_name.startswith('proj_'):

            # Construct the new folder name
            new_name = f"sampling_mesh_{dir_name}_mean"

            # Create full absolute paths
            old_path = os.path.join(root, dir_name)
            new_path = os.path.join(root, new_name)

            # Perform the rename
            try:
                os.rename(old_path, new_path)
                print(f"Renamed: {dir_name} -> {new_name}")
            except OSError as e:
                print(f"Error renaming {dir_name}: {e}")

print(">> Renaming Complete.")